这三种方法在数学建模中统称为**数据无量纲化（Dimensionless Processing）**或**标准化**。

在建立评价模型（如TOPSIS、熵权法、线性加权）之前，由于不同指标的单位不同（例如GDP是“亿元”，人均寿命是“岁”），数值量级差异极大，直接计算会导致大数吃小数。因此，我们需要通过数学变换将它们映射到同一个量级。

---

### 一、 极值差法 (Range Method / Min-Max Scaling)

这是建模中最常用的方法，将数据线性映射到 $[0, 1]$ 区间。

#### 1. 数学原理与推导
**核心思想**：计算当前值在整体波动范围（极差）中所处的位置比例。

**数学推导**：
设原始序列为 $X = \{x_1, x_2, \dots, x_n\}$。
*   **正向指标（效益型，越大越好）**：
    $$ y_i = \frac{x_i - \min(X)}{\max(X) - \min(X)} $$
*   **负向指标（成本型，越小越好）**：
    $$ y_i = \frac{\max(X) - x_i}{\max(X) - \min(X)} $$

**推导逻辑**：
通过减去最小值，将坐标原点移至 0；通过除以极差（最大减最小），将单位长度缩放为总长度的 1。

#### 2. 适用场景
*   **大多数综合评价模型**：如熵权法、TOPSIS、灰色关联分析。
*   **需求明确区间**：当你希望所有数据严格落在 0 到 1 之间时。
*   **缺点**：如果数据中有极端异常值（极大的噪声），会导致其他正常数据被压缩到非常窄的区间（如 0 到 0.01）。

#### 3. Python 代码
```python
import numpy as np

def range_scaling(data, direction=1):
    """
    极值差法
    direction: 1为正向, 0为负向
    """
    x_min = np.min(data)
    x_max = np.max(data)
    if direction == 1:
        return (data - x_min) / (x_max - x_min)
    else:
        return (x_max - data) / (x_max - x_min)
```

---

### 二、 标准差法 (Standard Deviation Method / Z-score)

这是统计学中最严谨的方法，处理后的数据均值为 0，标准差为 1。

#### 1. 数学原理与推导
**核心思想**：衡量原始数据偏离均值的程度（以标准差为单位）。

**数学推导**：
设均值为 $\mu$，标准差为 $\sigma$：
$$ \mu = \frac{1}{n} \sum x_i, \quad \sigma = \sqrt{\frac{1}{n} \sum (x_i - \mu)^2} $$
标准化公式：
$$ y_i = \frac{x_i - \mu}{\sigma} $$

**推导逻辑**：
1.  **中心化（Centering）**：$x_i - \mu$ 将数据平移，使中心点变为 0。
2.  **缩放（Scaling）**：除以 $\sigma$ 消除波动幅度的差异。

#### 2. 适用场景
*   **基于相关性的算法**：主成分分析（PCA）、聚类分析、回归分析。
*   **存在异常值**：相比极值差法，标准差法对异常值的敏感度较低，因为它利用了全局统计特性。
*   **注意**：处理后的数据会有负数，**不能直接用于熵权法**（熵权法要求数据必须为正）。

#### 3. Python 代码
```python
def z_score_scaling(data):
    return (data - np.mean(data)) / np.std(data)
```

---

### 三、 功效系数法 (Efficacy Coefficient Method)

这是一种带有“满意度”色彩的分数转化法，常用于业绩评价或综合评分。

#### 1. 数学原理与推导
**核心思想**：在极值差法的基础上，给定一个**满意区间**（如 60 分到 100 分），将指标转化为得分。

**数学推导**：
公式通常表示为：
$$ y_i = c + \frac{x_i - \min(X)}{\max(X) - \min(X)} \times d $$
*   $c$：**基础分**（如 60 分，代表及格线）。
*   $d$：**分值跨度**（如 40 分，使得总分最高为 60+40=100）。

**推导逻辑**：
它其实是极值差法的**仿射变换**。在评价模型中，它能保证每一个指标即使表现最差，也有一个“底线分”（保底分），符合人类的评价直觉。

#### 2. 适用场景
*   **多目标综合评价**：当你需要向非专业人士展示结果时，功效系数法得到的分数（如 85.5 分）比标准化后的 0.85 更有说服力。
*   **企业/项目考核**：设定具体的满意值和不允许值。

#### 3. Python 代码
```python
def efficacy_coefficient(data, c=60, d=40):
    """
    c: 基础分, d: 档次分
    映射结果区间为 [c, c+d]
    """
    x_min = np.min(data)
    x_max = np.max(data)
    # 极值差部分
    r = (data - x_min) / (x_max - x_min)
    # 仿射变换
    return c + r * d
```

---

### 四、 建模实战：论文中如何“修修补补”

在建模论文中，处理数据时可以参考以下话术：

1.  **方法选择的辩护**：
    *   “考虑到各评价指标在量纲和量级上存在显著差异，为确保模型的客观性，本文在建立综合评价体系前对原始数据进行了**无量纲化处理**。”
    *   “由于后续采用**熵权法**进行权重计算，为避免对数运算中出现零值及负值，本文选用了**极差法（Min-Max Scaling）**进行正向化和标准化处理。”

2.  **小技巧（避免 0 的出现）**：
    在极值差法中，最小值会变成 0。如果你的下一步算法要取对数（如熵权法 $\ln p_i$），$0$ 会导致程序报错。
    **修正公式**：
    $$ y_i = 0.002 + 0.996 \times \frac{x_i - \min(X)}{\max(X) - \min(X)} $$
    或者直接在结果上加一个极小值：`y = (data - min) / (max - min) + 0.0001`。这在论文中被称为**“平移修正”**。

3.  **逆向指标转化**：
    在评价类问题中，一定要先做**正向化**。如果你用标准差法，正向化需要手动：负向指标的正向化结果应该是 $-z$。

**下一组数据预处理技巧（如：独热编码、PCA降维推导、等宽/等频离散化），你想了解哪一个？**